# GuardCAN — Week-2 HPC engine on a Colab T4 (reproducibility fallback)

The **primary** results in `bench/hpc/` were measured on a local RTX 4060 Laptop GPU (WSL2). This notebook lets a
reviewer without a GPU rebuild the CUDA engine and re-run the **same** benchmarks on a free Colab T4.

What it does: train the Week-1 autoencoder (PyTorch, synthetic Car-Hacking-format data) → export the weights to
`.npz` → compile the `.cu` sources with `nvcc` → verify the GPU engine against the Python model → run the
memory-strategy, stream-pipeline and CPU-baseline benchmarks → chart the results.

**Honesty notes**
- Numbers from this notebook are *T4 numbers* and land in `bench/hpc-colab-t4/` — they never overwrite the
  RTX 4060 results. Expect them to differ: different GPU (sm_75, PCIe 3.0 on most Colab hosts), different CPU
  (2 shared vCPUs), and native Linux instead of WSL2 (so `unified+prefetch` runs there and the WSL2-specific
  stream-serialization effect may not appear).
- Colab VMs are shared; re-run if a result looks noisy. The benchmarks print min/max over repetitions.
- Input data is the synthetic Car-Hacking-format set (`data/make_synthetic.py`), replayed — labelled as such.

Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Get the code

In [ ]:
REPO = "https://github.com/casktech-sys/guardcan.git"   # or upload/unzip the repo and skip the clone
BRANCH = "main"
!git clone --depth 1 -b {BRANCH} {REPO} guardcan
%cd guardcan

In [ ]:
!pip install -q -r requirements.txt     # numpy/pandas/sklearn/matplotlib; PyTorch is preinstalled on Colab
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 2. Train the Week-1 autoencoder (PyTorch on the GPU) and export its weights

In [ ]:
!python run_week1.py                                                  # synthetic data -> train -> eval
!python -m model.export_weights --models models --data data/synthetic # -> models/ae_weights.npz + ae_reference.npz

## 3. Build the CUDA engine and verify it against the Python model

In [ ]:
!make -C engine ARCH=native
!make -C engine verify

## 4. Run the benchmarks (≈10–20 min on a T4)

Writes `cpu_baseline.json`, `infer_streams.json`, `mem_strategy.json` into `bench/hpc-colab-t4/`.

In [ ]:
OUT = "bench/hpc-colab-t4"
!make -C engine bench OUT=../{OUT}

## 5. (Optional) Nsight Systems timeline

`nsys` is not always present on Colab images. If the cell below reports it missing, install the CLI from NVIDIA's
CUDA apt repository (e.g. `apt-get install -y nsight-systems-2025.5.2`) or skip this step — the charts that do not
need the trace are still produced.

In [ ]:
import shutil, subprocess
if shutil.which("nsys"):
    !make -C engine nsys OUT=../{OUT}
else:
    print("nsys not found on this runtime — skipping the timeline capture")

## 6. Charts and tables

In [ ]:
!python -m engine.plot_results --dir {OUT}
from IPython.display import Image, Markdown, display
import os
for png in ["mem_strategy_throughput.png", "mem_strategy_latency.png", "streams_sweep.png", "nsys_timeline.png"]:
    p = os.path.join(OUT, png)
    if os.path.exists(p):
        display(Image(p))
display(Markdown(open(os.path.join(OUT, "results.md")).read()))

## 7. Compare with the local RTX 4060 results

The committed local results are in `bench/hpc/` (`results.md`, `summary.json`). When comparing, keep the platform
differences above in mind — the point of this notebook is that the pipeline builds, verifies and measures anywhere,
not that a T4 should match a laptop RTX 4060.